# 00 — Dataset Preparation

This notebook covers:
1. Downloading NYC Yellow Taxi Trip Records (starting with January 2009)
2. Exploring and cleaning the raw data
3. Normalising column names to the blog convention (`fare_amt`, `tip_amt`, …)
4. Converting CSV → Parquet for efficient benchmarking
5. Describing the two additional datasets (one smaller, one larger)

**Blog reference:** *Benchmark: Koalas (PySpark) and Dask* by Xinrong Meng & Hyukjin Kwon.

## 1  Environment setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config import (
    CSV_DIR, PARQUET_DIR, RESULTS_DIR, TLC_CDN_BASE,
    COLUMN_RENAME, parquet_path,
)

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy :', np.__version__)

## 2  Download: NYC Yellow Taxi — January 2009

We start with a **single month** as the blog recommends for initial experiments,
then scale up by adding more months.

Data source: [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [ ]:
# Run the download script from here, or call it from the command line:
#   python scripts/download_data.py --year 2009 --months 1
#
# To add more months (scale up):
#   python scripts/download_data.py --year 2009 --months 1 2 3 4 5 6

import subprocess, shlex
cmd = 'python ../scripts/download_data.py --year 2009 --months 1'
result = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr)

In [ ]:
# Confirm the Parquet file exists and show its size
pq_path = parquet_path('yellow_taxi.parquet')
if os.path.exists(pq_path):
    size_mb = os.path.getsize(pq_path) / 1e6
    print(f'Parquet file: {pq_path}')
    print(f'Size        : {size_mb:.1f} MB')
else:
    print(f'File not found: {pq_path}')
    print('Check that the download step above completed successfully.')

## 3  Exploratory Data Analysis

In [ ]:
df = pd.read_parquet(pq_path)
print(f'Shape : {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head(3)

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else '  None')

In [ ]:
# Distribution of the target variable (fare_amt)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_plot = df[df['fare_amt'].between(0, 100)]
axes[0].hist(df_plot['fare_amt'], bins=50, edgecolor='black')
axes[0].set(title='fare_amt distribution', xlabel='fare_amt ($)', ylabel='count')
axes[1].hist(df_plot['tip_amt'], bins=50, edgecolor='black', color='orange')
axes[1].set(title='tip_amt distribution', xlabel='tip_amt ($)', ylabel='count')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fare_tip_distribution.png'), dpi=120)
plt.show()

In [ ]:
# Vendor distribution
print('vendor_name value counts:')
print(df['vendor_name'].value_counts())
print()
print('payment_type value counts:')
print(df['payment_type'].value_counts())

In [ ]:
# Blog filter impact: tip_amt in [1, 5)
df_filtered = df[(df['tip_amt'] >= 1) & (df['tip_amt'] < 5)]
pct = 100 * len(df_filtered) / len(df)
print(f'Rows after tip_amt filter [1,5): {len(df_filtered):,}  ({pct:.1f}% of original)')
# Note: the blog reports this filter keeps ~36% of rows

## 4  Column Normalisation Summary

The 2009 TLC CSV uses mixed-case column names (`Fare_Amt`, `Tip_Amt`, etc.).
We map them to the lowercase names used throughout the blog benchmark.

| Raw column (2009 CSV) | Normalised name | Description |
|---|---|---|
| `Fare_Amt` | `fare_amt` | Meter-registered fare (target for ML) |
| `Tip_Amt` | `tip_amt` | Tip amount (used for filtering) |
| `vendor_name` | `vendor_name` | Vendor ID (used for groupby / join) |
| `Trip_Distance` | `trip_distance` | Trip distance in miles |
| `Payment_Type` | `payment_type` | Payment method (used for value_counts) |
| `Passenger_Count` | `passenger_count` | Number of passengers |

In [ ]:
# Verify normalised columns are present
required = ['fare_amt', 'tip_amt', 'vendor_name', 'trip_distance', 'payment_type']
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    print(f'WARNING: missing columns: {missing_cols}')
else:
    print('All required columns present:', required)

## 5  Scaling up: adding more months

The blog used 2009-2013 data (157 GB).  We scale up incrementally:

| File | Approx. rows | Approx. size (CSV) |
|---|---|---|
| `yellow_tripdata_2009-01` | ~14 M | ~1.4 GB |
| 2009 full year (12 months) | ~170 M | ~16 GB |
| 2009–2011 (3 years) | ~500 M | ~50 GB |
| 2009–2013 (5 years, blog) | ~800 M | ~157 GB |

Run the download script with `--months 1 2 3 … 12` to expand.

## 6  Additional Datasets

The project requires two extra datasets — one smaller and one larger than the single-month taxi file.

### 6a  Smaller dataset — California Housing (sklearn)
~20 K rows, 8 numeric features.  Used to verify that pipelines work before scaling.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame
print('California Housing shape:', df_housing.shape)
print(df_housing.head(3))

# Save as Parquet
housing_path = parquet_path('california_housing.parquet')
df_housing.to_parquet(housing_path, index=False)
print(f'Saved to: {housing_path}')

### 6b  Larger dataset — NYC Taxi full 2009 year (or multi-year)

Download all 12 months of 2009 to get a dataset ~12× larger than the single-month file.
This is enough to observe distributed-compute benefits without consuming all GCP credits.

In [ ]:
# Uncomment and run when you want to download the full 2009 year:
# cmd = 'python ../scripts/download_data.py --year 2009 --months 1 2 3 4 5 6 7 8 9 10 11 12 --output yellow_taxi_2009_full.parquet'
# result = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
# print(result.stdout)

## 7  Dataset Summary Table

| Dataset | File | Rows (approx) | Columns | ML Task |
|---|---|---|---|---|
| California Housing | `california_housing.parquet` | 20,640 | 9 | Regression (median house value) |
| NYC Yellow Taxi Jan 2009 | `yellow_taxi.parquet` | ~14 M | 16 | Regression (fare_amt) |
| NYC Yellow Taxi 2009 (full year) | `yellow_taxi_2009_full.parquet` | ~170 M | 16 | Regression (fare_amt) |

In [ ]:
# Quick summary stats for the benchmark dataset
print('=== NYC Taxi Jan 2009 ===')
print(df[['fare_amt', 'tip_amt', 'trip_distance', 'passenger_count']].describe().round(2))